# Bank Management System Using Python and MySQL

## Project Description

A menu-driven banking application developed using Python and MySQL.

## Technologies Used

- Python
- MySQL
- MySQL Connector
- Jupyter Notebook

## Features

1. Create Account
2. View Account
3. Deposit Money
4. Withdraw Money
5. Check Balance
6. Transfer Money
7. Transaction History
8. Update Customer Details
9. Close Account
10. Exit

In [2]:
!pip install mysql-connector-python

   ---------------------------------------- 0.0/17.7 MB ? eta -:--:--
   ---------------------------------------- 0.0/17.7 MB 682.7 kB/s eta 0:00:26
   ---------------------------------------- 0.0/17.7 MB 653.6 kB/s eta 0:00:27
   ---------------------------------------- 0.1/17.7 MB 751.6 kB/s eta 0:00:24
   ---------------------------------------- 0.1/17.7 MB 751.6 kB/s eta 0:00:24
   ---------------------------------------- 0.2/17.7 MB 807.1 kB/s eta 0:00:22
    --------------------------------------- 0.3/17.7 MB 1.1 MB/s eta 0:00:16
    --------------------------------------- 0.4/17.7 MB 1.3 MB/s eta 0:00:14
   - -------------------------------------- 0.5/17.7 MB 1.3 MB/s eta 0:00:14
   - -------------------------------------- 0.6/17.7 MB 1.6 MB/s eta 0:00:11
   - -------------------------------------- 0.7/17.7 MB 1.6 MB/s eta 0:00:11
   - -------------------------------------- 0.8/17.7 MB 1.6 MB/s eta 0:00:11
   -- ------------------------------------- 1.0/17.7 MB 1.9 MB/s eta 0:00

In [24]:
import mysql.connector
print("MySQL Connector is installed successfully!")

MySQL Connector is installed successfully!


In [25]:
import mysql.connector

connection = mysql.connector.connect(
    host="localhost",
    user="root",
    password="YOUR_PASSWORD"
)

cursor = connection.cursor()

cursor.execute("CREATE DATABASE IF NOT EXISTS BankDB")

print("BankDB database created successfully!")

BankDB database created successfully!


In [26]:
connection = mysql.connector.connect(
    host="localhost",
    user="root",
    password="YOUR_PASSWORD",
    database="BankDB"
)

cursor = connection.cursor()

print("Connected to BankDB successfully!")

Connected to BankDB successfully!


In [27]:
cursor.execute("""
CREATE TABLE IF NOT EXISTS Customers (
    customer_id INT AUTO_INCREMENT PRIMARY KEY,
    name VARCHAR(100) NOT NULL,
    phone VARCHAR(15),
    email VARCHAR(100),
    address VARCHAR(200)
)
""")

connection.commit()

print("Customers table created successfully!")

Customers table created successfully!


In [28]:
cursor.execute("""
CREATE TABLE IF NOT EXISTS Accounts (
    account_no BIGINT PRIMARY KEY,
    customer_id INT,
    account_type VARCHAR(20),
    balance DECIMAL(12,2) DEFAULT 0.00,
    created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
    FOREIGN KEY (customer_id) REFERENCES Customers(customer_id)
)
""")

connection.commit()

print("Accounts table created successfully!")

Accounts table created successfully!


In [29]:
cursor.execute("""
CREATE TABLE IF NOT EXISTS Transactions (
    transaction_id INT AUTO_INCREMENT PRIMARY KEY,
    account_no BIGINT,
    transaction_type VARCHAR(20),
    amount DECIMAL(12,2),
    transaction_date TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
    FOREIGN KEY (account_no) REFERENCES Accounts(account_no)
)
""")

connection.commit()

print("Transactions table created successfully!")

Transactions table created successfully!


In [30]:
cursor.execute("SHOW TABLES")

for table in cursor:
    print(table[0])

accounts
customers
transactions


In [31]:
import random

def create_account():
    print("----- CREATE NEW ACCOUNT -----")
    
    name = input("Enter customer name: ")
    phone = input("Enter phone number: ")
    email = input("Enter email: ")
    address = input("Enter address: ")
    
    account_type = input("Enter account type (Savings/Current): ")
    initial_deposit = float(input("Enter initial deposit amount: "))
    
    # Generate a 10-digit account number
    account_no = random.randint(1000000000, 9999999999)
    
    # Insert customer details
    customer_query = """
    INSERT INTO Customers (name, phone, email, address)
    VALUES (%s, %s, %s, %s)
    """
    
    cursor.execute(customer_query, (name, phone, email, address))
    connection.commit()
    
    # Get the newly created customer ID
    customer_id = cursor.lastrowid
    
    # Insert account details
    account_query = """
    INSERT INTO Accounts 
    (account_no, customer_id, account_type, balance)
    VALUES (%s, %s, %s, %s)
    """
    
    cursor.execute(
        account_query,
        (account_no, customer_id, account_type, initial_deposit)
    )
    connection.commit()
    
    print("\nAccount created successfully!")
    print("Customer ID:", customer_id)
    print("Account Number:", account_no)
    print("Account Type:", account_type)
    print("Initial Balance: ₹", initial_deposit)

In [32]:
create_account()

----- CREATE NEW ACCOUNT -----


Enter customer name:  Harshitha
Enter phone number:  9876543210
Enter email:  harshitha@gmail.com
Enter address:  Hyderabad
Enter account type (Savings/Current):  Savings
Enter initial deposit amount:  5000



Account created successfully!
Customer ID: 2
Account Number: 5072189817
Account Type: Savings
Initial Balance: ₹ 5000.0


In [33]:
def view_account():
    print("----- VIEW ACCOUNT -----")
    
    account_no = input("Enter account number: ")

    query = """
    SELECT 
        c.customer_id,
        c.name,
        c.phone,
        c.email,
        c.address,
        a.account_no,
        a.account_type,
        a.balance,
        a.created_at
    FROM Customers c
    JOIN Accounts a
        ON c.customer_id = a.customer_id
    WHERE a.account_no = %s
    """

    cursor.execute(query, (account_no,))
    account = cursor.fetchone()

    if account:
        print("\n----- ACCOUNT DETAILS -----")
        print("Customer ID   :", account[0])
        print("Name          :", account[1])
        print("Phone         :", account[2])
        print("Email         :", account[3])
        print("Address       :", account[4])
        print("Account No    :", account[5])
        print("Account Type  :", account[6])
        print("Balance       : ₹", account[7])
        print("Created Date  :", account[8])
    else:
        print("Account not found.")

In [34]:
view_account()

----- VIEW ACCOUNT -----


Enter account number:  5072189817



----- ACCOUNT DETAILS -----
Customer ID   : 2
Name          : Harshitha
Phone         : 9876543210
Email         : harshitha@gmail.com
Address       : Hyderabad
Account No    : 5072189817
Account Type  : Savings
Balance       : ₹ 5000.00
Created Date  : 2026-09-03 12:20:55


In [37]:
def deposit_money():
    print("----- DEPOSIT MONEY -----")
    
    account_no = input("Enter account number: ")
    amount = float(input("Enter deposit amount: ₹"))

    if amount <= 0:
        print("Please enter a valid amount.")
        return

    # Check whether the account exists
    cursor.execute(
        "SELECT balance FROM Accounts WHERE account_no = %s",
        (account_no,)
    )

    account = cursor.fetchone()

    if account is None:
        print("Account not found.")
        return

    # Update balance
    cursor.execute(
        """
        UPDATE Accounts
        SET balance = balance + %s
        WHERE account_no = %s
        """,
        (amount, account_no)
    )

    # Record transaction
    cursor.execute(
        """
        INSERT INTO Transactions
        (account_no, transaction_type, amount)
        VALUES (%s, %s, %s)
        """,
        (account_no, "Deposit", amount)
    )

    connection.commit()

    # Get updated balance directly from MySQL
    cursor.execute(
        "SELECT balance FROM Accounts WHERE account_no = %s",
        (account_no,)
    )

    new_balance = cursor.fetchone()[0]

    print("\nDeposit successful!")
    print("Deposited Amount: ₹", amount)
    print("New Balance: ₹", new_balance)

In [38]:
deposit_money()

----- DEPOSIT MONEY -----


Enter account number:  5072189817
Enter deposit amount: ₹ 2000



Deposit successful!
Deposited Amount: ₹ 2000.0
New Balance: ₹ 9000.00


In [39]:
def withdraw_money():
    print("----- WITHDRAW MONEY -----")
    
    account_no = input("Enter account number: ")
    amount = float(input("Enter withdrawal amount: ₹"))

    if amount <= 0:
        print("Please enter a valid amount.")
        return

    # Check account and balance
    cursor.execute(
        "SELECT balance FROM Accounts WHERE account_no = %s",
        (account_no,)
    )

    account = cursor.fetchone()

    if account is None:
        print("Account not found.")
        return

    balance = float(account[0])

    # Check sufficient balance
    if amount > balance:
        print("Insufficient balance.")
        print("Available Balance: ₹", balance)
        return

    # Update balance
    cursor.execute(
        """
        UPDATE Accounts
        SET balance = balance - %s
        WHERE account_no = %s
        """,
        (amount, account_no)
    )

    # Record transaction
    cursor.execute(
        """
        INSERT INTO Transactions
        (account_no, transaction_type, amount)
        VALUES (%s, %s, %s)
        """,
        (account_no, "Withdrawal", amount)
    )

    connection.commit()

    # Get updated balance
    cursor.execute(
        "SELECT balance FROM Accounts WHERE account_no = %s",
        (account_no,)
    )

    new_balance = cursor.fetchone()[0]

    print("\nWithdrawal successful!")
    print("Withdrawn Amount: ₹", amount)
    print("New Balance: ₹", new_balance)

In [40]:
withdraw_money()

----- WITHDRAW MONEY -----


Enter account number:  5072189817
Enter withdrawal amount: ₹ 5000



Withdrawal successful!
Withdrawn Amount: ₹ 5000.0
New Balance: ₹ 4000.00


In [41]:
def transaction_history():
    print("----- TRANSACTION HISTORY -----")
    
    account_no = input("Enter account number: ")

    # Check whether the account exists
    cursor.execute(
        "SELECT account_no FROM Accounts WHERE account_no = %s",
        (account_no,)
    )

    account = cursor.fetchone()

    if account is None:
        print("Account not found.")
        return

    # Get all transactions
    cursor.execute(
        """
        SELECT transaction_id, transaction_type, amount, transaction_date
        FROM Transactions
        WHERE account_no = %s
        ORDER BY transaction_date
        """,
        (account_no,)
    )

    transactions = cursor.fetchall()

    if not transactions:
        print("No transactions found.")
        return

    print("\n----- TRANSACTIONS -----")

    for transaction in transactions:
        print("Transaction ID :", transaction[0])
        print("Type           :", transaction[1])
        print("Amount         : ₹", transaction[2])
        print("Date           :", transaction[3])
        print("------------------------------")

In [42]:
transaction_history()

----- TRANSACTION HISTORY -----


Enter account number:  5072189817



----- TRANSACTIONS -----
Transaction ID : 4
Type           : Deposit
Amount         : ₹ 2000.00
Date           : 2026-09-03 12:23:41
------------------------------
Transaction ID : 5
Type           : Deposit
Amount         : ₹ 2000.00
Date           : 2026-09-03 12:24:51
------------------------------
Transaction ID : 6
Type           : Withdrawal
Amount         : ₹ 5000.00
Date           : 2026-09-03 12:26:50
------------------------------


In [43]:
cursor.execute("""
SELECT 
    a.account_no,
    c.name,
    a.account_type,
    a.balance
FROM Accounts a
JOIN Customers c
ON a.customer_id = c.customer_id
ORDER BY a.created_at
""")

accounts = cursor.fetchall()

for account in accounts:
    print("Account Number :", account[0])
    print("Name           :", account[1])
    print("Account Type   :", account[2])
    print("Balance        : ₹", account[3])
    print("-----------------------------")

Account Number : 8911873937
Name           : Harshitha
Account Type   : Savings
Balance        : ₹ 19000.00
-----------------------------
Account Number : 5072189817
Name           : Harshitha
Account Type   : Savings
Balance        : ₹ 4000.00
-----------------------------


In [44]:
cursor.execute("SELECT account_no FROM Accounts")

for account in cursor.fetchall():
    print(account[0])

8911873937
5072189817


In [45]:
def check_balance():
    print("----- CHECK BALANCE -----")
    
    account_no = input("Enter account number: ")

    cursor.execute(
        "SELECT balance FROM Accounts WHERE account_no = %s",
        (account_no,)
    )

    account = cursor.fetchone()

    if account is None:
        print("Account not found.")
        return

    print("\nAccount Number :", account_no)
    print("Current Balance: ₹", account[0])

In [46]:
check_balance()

----- CHECK BALANCE -----


Enter account number:  5072189817



Account Number : 5072189817
Current Balance: ₹ 4000.00


In [47]:
def transfer_money():
    print("----- TRANSFER MONEY -----")

    sender_account = input("Enter sender account number: ")
    receiver_account = input("Enter receiver account number: ")

    if sender_account == receiver_account:
        print("Sender and receiver accounts cannot be the same.")
        return

    try:
        amount = float(input("Enter transfer amount: ₹"))
    except ValueError:
        print("Please enter a valid amount.")
        return

    if amount <= 0:
        print("Transfer amount must be greater than zero.")
        return

    # Check sender account
    cursor.execute(
        "SELECT balance FROM Accounts WHERE account_no = %s",
        (sender_account,)
    )
    sender = cursor.fetchone()

    if sender is None:
        print("Sender account not found.")
        return

    # Check receiver account
    cursor.execute(
        "SELECT account_no FROM Accounts WHERE account_no = %s",
        (receiver_account,)
    )
    receiver = cursor.fetchone()

    if receiver is None:
        print("Receiver account not found.")
        return

    # Check sender balance
    sender_balance = float(sender[0])

    if amount > sender_balance:
        print("Insufficient balance.")
        print("Available Balance: ₹", sender_balance)
        return

    try:
        # Deduct from sender
        cursor.execute(
            """
            UPDATE Accounts
            SET balance = balance - %s
            WHERE account_no = %s
            """,
            (amount, sender_account)
        )

        # Add to receiver
        cursor.execute(
            """
            UPDATE Accounts
            SET balance = balance + %s
            WHERE account_no = %s
            """,
            (amount, receiver_account)
        )

        # Record sender transaction
        cursor.execute(
            """
            INSERT INTO Transactions
            (account_no, transaction_type, amount)
            VALUES (%s, %s, %s)
            """,
            (sender_account, "Transfer Sent", amount)
        )

        # Record receiver transaction
        cursor.execute(
            """
            INSERT INTO Transactions
            (account_no, transaction_type, amount)
            VALUES (%s, %s, %s)
            """,
            (receiver_account, "Transfer Received", amount)
        )

        connection.commit()

        print("\nTransfer successful!")
        print("Transferred Amount: ₹", amount)
        print("From Account      :", sender_account)
        print("To Account        :", receiver_account)

    except Exception as e:
        connection.rollback()
        print("Transfer failed:", e)

In [48]:
transfer_money()

----- TRANSFER MONEY -----


Enter sender account number:  5072189817
Enter receiver account number:  8911873937
Enter transfer amount: ₹ 1000



Transfer successful!
Transferred Amount: ₹ 1000.0
From Account      : 5072189817
To Account        : 8911873937


In [49]:
check_balance()

----- CHECK BALANCE -----


Enter account number:  5072189817



Account Number : 5072189817
Current Balance: ₹ 3000.00


In [50]:
check_balance()

----- CHECK BALANCE -----


Enter account number:  8911873937



Account Number : 8911873937
Current Balance: ₹ 20000.00


In [51]:
transaction_history()

----- TRANSACTION HISTORY -----


Enter account number:  5072189817



----- TRANSACTIONS -----
Transaction ID : 4
Type           : Deposit
Amount         : ₹ 2000.00
Date           : 2026-09-03 12:23:41
------------------------------
Transaction ID : 5
Type           : Deposit
Amount         : ₹ 2000.00
Date           : 2026-09-03 12:24:51
------------------------------
Transaction ID : 6
Type           : Withdrawal
Amount         : ₹ 5000.00
Date           : 2026-09-03 12:26:50
------------------------------
Transaction ID : 7
Type           : Transfer Sent
Amount         : ₹ 1000.00
Date           : 2026-09-03 12:36:55
------------------------------


In [52]:
transaction_history()

----- TRANSACTION HISTORY -----


Enter account number:  8911873937



----- TRANSACTIONS -----
Transaction ID : 1
Type           : Deposit
Amount         : ₹ 10000.00
Date           : 2026-09-03 12:05:58
------------------------------
Transaction ID : 2
Type           : Deposit
Amount         : ₹ 2000.00
Date           : 2026-09-03 12:10:22
------------------------------
Transaction ID : 3
Type           : Deposit
Amount         : ₹ 2000.00
Date           : 2026-09-03 12:11:46
------------------------------
Transaction ID : 8
Type           : Transfer Received
Amount         : ₹ 1000.00
Date           : 2026-09-03 12:36:55
------------------------------


In [53]:
def update_customer():
    print("----- UPDATE CUSTOMER DETAILS -----")

    account_no = input("Enter account number: ")

    # Find customer linked to the account
    cursor.execute("""
        SELECT c.customer_id, c.name, c.phone, c.email, c.address
        FROM Customers c
        JOIN Accounts a
        ON c.customer_id = a.customer_id
        WHERE a.account_no = %s
    """, (account_no,))

    customer = cursor.fetchone()

    if customer is None:
        print("Account not found.")
        return

    customer_id = customer[0]

    print("\nCurrent Details:")
    print("Name    :", customer[1])
    print("Phone   :", customer[2])
    print("Email   :", customer[3])
    print("Address :", customer[4])

    print("\nEnter new details:")

    phone = input("Enter new phone number: ")
    email = input("Enter new email: ")
    address = input("Enter new address: ")

    cursor.execute("""
        UPDATE Customers
        SET phone = %s,
            email = %s,
            address = %s
        WHERE customer_id = %s
    """, (phone, email, address, customer_id))

    connection.commit()

    print("\nCustomer details updated successfully!")

In [54]:
update_customer()

----- UPDATE CUSTOMER DETAILS -----


Enter account number:  5072189817



Current Details:
Name    : Harshitha
Phone   : 9876543210
Email   : harshitha@gmail.com
Address : Hyderabad

Enter new details:


Enter new phone number:  9999999999
Enter new email:  harshithanew@gmail.com
Enter new address:  Secunderabad



Customer details updated successfully!


In [55]:
view_account()

----- VIEW ACCOUNT -----


Enter account number:  5072189817



----- ACCOUNT DETAILS -----
Customer ID   : 2
Name          : Harshitha
Phone         : 9999999999
Email         : harshithanew@gmail.com
Address       : Secunderabad
Account No    : 5072189817
Account Type  : Savings
Balance       : ₹ 3000.00
Created Date  : 2026-09-03 12:20:55


In [56]:
transfer_money()

----- TRANSFER MONEY -----


Enter sender account number:  5072189817
Enter receiver account number:  8911873937
Enter transfer amount: ₹ 3000



Transfer successful!
Transferred Amount: ₹ 3000.0
From Account      : 5072189817
To Account        : 8911873937


In [57]:
view_account()

----- VIEW ACCOUNT -----


Enter account number:  5072189817



----- ACCOUNT DETAILS -----
Customer ID   : 2
Name          : Harshitha
Phone         : 9999999999
Email         : harshithanew@gmail.com
Address       : Secunderabad
Account No    : 5072189817
Account Type  : Savings
Balance       : ₹ 0.00
Created Date  : 2026-09-03 12:20:55


In [58]:
def close_account():
    print("----- CLOSE BANK ACCOUNT -----")

    account_no = input("Enter account number: ")

    # Check account and balance
    cursor.execute("""
        SELECT customer_id, balance
        FROM Accounts
        WHERE account_no = %s
    """, (account_no,))

    account = cursor.fetchone()

    if account is None:
        print("Account not found.")
        return

    customer_id = account[0]
    balance = float(account[1])

    print("\nCurrent Balance: ₹", balance)

    # Do not allow closing if money remains
    if balance != 0:
        print("Cannot close the account.")
        print("Please withdraw or transfer the remaining balance first.")
        return

    confirmation = input(
        "Are you sure you want to close this account? (yes/no): "
    )

    if confirmation.lower() != "yes":
        print("Account closure cancelled.")
        return

    try:
        # Delete transactions first
        cursor.execute("""
            DELETE FROM Transactions
            WHERE account_no = %s
        """, (account_no,))

        # Delete account
        cursor.execute("""
            DELETE FROM Accounts
            WHERE account_no = %s
        """, (account_no,))

        # Delete customer
        cursor.execute("""
            DELETE FROM Customers
            WHERE customer_id = %s
        """, (customer_id,))

        connection.commit()

        print("\nAccount closed successfully!")

    except Exception as e:
        connection.rollback()
        print("Account closure failed:", e)

In [59]:
close_account()

----- CLOSE BANK ACCOUNT -----


Enter account number:  5072189817



Current Balance: ₹ 0.0


Are you sure you want to close this account? (yes/no):  yes



Account closed successfully!


In [60]:
def main_menu():
    while True:
        print("\n")
        print("=" * 40)
        print("       BANK MANAGEMENT SYSTEM")
        print("=" * 40)
        print("1. Create Account")
        print("2. View Account")
        print("3. Deposit Money")
        print("4. Withdraw Money")
        print("5. Check Balance")
        print("6. Transfer Money")
        print("7. Transaction History")
        print("8. Update Customer Details")
        print("9. Close Account")
        print("10. Exit")
        print("=" * 40)

        choice = input("Enter your choice (1-10): ")

        if choice == "1":
            create_account()

        elif choice == "2":
            view_account()

        elif choice == "3":
            deposit_money()

        elif choice == "4":
            withdraw_money()

        elif choice == "5":
            check_balance()

        elif choice == "6":
            transfer_money()

        elif choice == "7":
            transaction_history()

        elif choice == "8":
            update_customer()

        elif choice == "9":
            close_account()

        elif choice == "10":
            print("\nThank you for using Bank Management System!")
            break

        else:
            print("\nInvalid choice. Please select a number from 1 to 10.")

In [61]:
main_menu()



       BANK MANAGEMENT SYSTEM
1. Create Account
2. View Account
3. Deposit Money
4. Withdraw Money
5. Check Balance
6. Transfer Money
7. Transaction History
8. Update Customer Details
9. Close Account
10. Exit


Enter your choice (1-10):  5


----- CHECK BALANCE -----


Enter account number:  5072189817


Account not found.


       BANK MANAGEMENT SYSTEM
1. Create Account
2. View Account
3. Deposit Money
4. Withdraw Money
5. Check Balance
6. Transfer Money
7. Transaction History
8. Update Customer Details
9. Close Account
10. Exit


...

----- CHECK BALANCE -----


Enter account number:  8911873937



Account Number : 8911873937
Current Balance: ₹ 23000.00


       BANK MANAGEMENT SYSTEM
1. Create Account
2. View Account
3. Deposit Money
4. Withdraw Money
5. Check Balance
6. Transfer Money
7. Transaction History
8. Update Customer Details
9. Close Account
10. Exit


Enter your choice (1-10):  10



Thank you for using Bank Management System!
